# CSE 842 — Hallucination Detection Baseline

Research question: Can we identify unsupported claims in AI-generated text by comparing it against the original source document?

Data: RAGTruth, news-summary subset. Each row is one LLM-written summary of a news article, labeled 1 if annotators found any hallucinated text and 0 otherwise.

## 1. Load data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

responses = pd.read_json("response.jsonl", lines=True)  # LLM outputs + hallucination labels
sources = pd.read_json("source_info.jsonl", lines=True)  # original articles

## 2. Prepare data

We keep only news summaries and use RAGTruth's official train split. The official test split is saved for the final paper. We split train 80/20 into train/validation by article, so the same article never appears in both.

In [ ]:
# attach each summary to its source article and keep news summaries only
df = responses.merge(sources, on="source_id")
df = df[df["task_type"] == "Summary"].copy()
df["article"] = df["source_info"]
df["summary"] = df["response"]

# label: 1 if any hallucinated span was annotated, else 0
df["hallucination"] = df["labels"].apply(len).gt(0).astype(int)

# train/validation split, grouped by article
train_pool = df[df["split"] == "train"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(train_pool, groups=train_pool["source_id"]))
train = train_pool.iloc[train_idx]
val = train_pool.iloc[val_idx]

print("Train:", len(train), "| Validation:", len(val))
print("Label counts (0 = clean, 1 = hallucinated):", train_pool["hallucination"].value_counts().to_dict())

Train: 3804 | Validation: 954
Label counts (0 = clean, 1 = hallucinated): {0: 3276, 1: 1482}


## 3. Features

Two simple features per summary:
- Similarity: TF-IDF cosine similarity between article and summary (word overlap).
- Length ratio: summary length / article length.

These shallow features can't detect contradictions, which is why this is a baseline.

In [ ]:
# learn vocabulary from training text only
vectorizer = TfidfVectorizer(stop_words="english", max_features=10000)
vectorizer.fit(pd.concat([train["article"], train["summary"]]))


def make_features(data):
    article_vecs = vectorizer.transform(data["article"])
    summary_vecs = vectorizer.transform(data["summary"])
    similarity = np.asarray(article_vecs.multiply(summary_vecs).sum(axis=1)).ravel()
    length_ratio = data["summary"].str.len() / (data["article"].str.len() + 1)
    return np.column_stack([similarity, length_ratio])


X_train, y_train = make_features(train), train["hallucination"]
X_val, y_val = make_features(val), val["hallucination"]

## 4. Models

- Majority class: always predicts "no hallucination" (reference floor).
- Logistic regression: uses the two features; `class_weight="balanced"` compensates for fewer hallucinated examples.

In [4]:
models = {
    "Majority class": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
}

for model in models.values():
    model.fit(X_train, y_train)

## 5. Validation results

Hallucinated = positive class.

In [5]:
rows = []
for name, model in models.items():
    pred = model.predict(X_val)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred, zero_division=0),
        "Recall": recall_score(y_val, pred, zero_division=0),
        "F1": f1_score(y_val, pred, zero_division=0),
    })

results = pd.DataFrame(rows).round(3)
results

,Model,Accuracy,Precision,Recall,F1
0,Majority class,0.718,0.000,0.000,0.000
1,Logistic Regression,0.483,0.304,0.647,0.414
